In [13]:
pip install mplfinance

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mplfinance as mpf

In [15]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
H1 = pd.read_csv(
    '/content/drive/MyDrive/H1_timeframe.csv',
    parse_dates=['time'],
    index_col='time'
)


In [21]:
def standardize_ohlc(H1):
    return H1.rename(columns={
        "o": "Open",
        "h": "High",
        "l": "Low",
        "c": "Close",
        "volume": "Volume"
    })

H1    = standardize_ohlc(H1)


In [22]:
H1.head()


,Open,High,Low,Close,Volume,complete
time,,,,,,
2015-01-01 23:00:00,1183.371,1187.389,1182.920,1186.728,642,True
2015-01-02 00:00:00,1186.681,1188.220,1186.121,1187.366,570,True
2015-01-02 01:00:00,1187.346,1187.346,1184.270,1184.965,418,True
2015-01-02 02:00:00,1185.008,1185.385,1184.206,1185.028,228,True
2015-01-02 03:00:00,1184.986,1186.105,1184.435,1186.105,223,True


In [23]:
# Candle features
H1['Body'] = (H1['Close'] - H1['Open']).abs()
H1['UpperWick'] = H1['High'] - H1[['Close', 'Open']].max(axis=1)
H1['LowerWick'] = H1[['Close', 'Open']].min(axis=1) - H1['Low']
H1['TotalRange'] = H1['High'] - H1['Low']

H1['PrevClose'] = H1['Close'].shift(1)
H1['HL'] = H1['High'] - H1['Low']
H1['HC'] = (H1['High'] - H1['PrevClose']).abs()
H1['LC'] = (H1['Low'] - H1['PrevClose']).abs()
H1['TrueRange'] = H1[['HL', 'HC', 'LC']].max(axis=1)
H1.drop(columns=['HL', 'HC', 'LC'], inplace=True)

# Volatility features
H1['ATR_14'] = H1['TrueRange'].rolling(14).mean()
H1['ATR_20'] = H1['TrueRange'].rolling(20).mean()
H1['ATR_50'] = H1['TrueRange'].rolling(50).mean()
H1['RollingVol'] = H1['Close'].pct_change().rolling(20).std()

# Time features
H1['Hour'] = H1.index.hour
H1['Session'] = pd.cut(H1['Hour'],
                       bins=[0, 8, 16, 24],
                       labels=['Asia', 'London', 'NY'])


In [24]:
"""
Swing High (pivot high): the High is the maximum in a window of (left + right + 1) candles

Swing Low (pivot low): the Low is the minimum in that window

"""

def add_pivots(df, left=3, right=3):
    w = left + right + 1

    pivot_high = df['High'].eq(df['High'].rolling(w, center=True).max())
    pivot_low  = df['Low'].eq(df['Low'].rolling(w, center=True).min())

    df['PivotHigh'] = pivot_high.fillna(False)
    df['PivotLow']  = pivot_low.fillna(False)
    return df


In [25]:
"""

Swing high = highest point that caused the low

Swing low = lowest point that caused the high

- A pivot high “caused the low” if price drops by at least k * ATR within the next N candles

- A pivot low “caused the high” if price rises by at least k * ATR within the next N candles

"""


def confirm_swings(df, lookahead=20, atr_col='ATR_14', atr_mult=1.0):
    # future lowest low and highest high over the next lookahead candles
    future_low  = df['Low'].shift(-1).rolling(lookahead).min()
    future_high = df['High'].shift(-1).rolling(lookahead).max()

    # how far price moves after the pivot
    drop_from_high = df['High'] - future_low
    rise_from_low  = future_high - df['Low']

    df['SwingHigh'] = df['PivotHigh'] & (drop_from_high >= atr_mult * df[atr_col])
    df['SwingLow']  = df['PivotLow']  & (rise_from_low  >= atr_mult * df[atr_col])
    return df


In [27]:
"""
Uptrend: latest confirmed swings show Higher Highs (HH) and Higher Lows (HL)

Downtrend: latest confirmed swings show Lower Highs (LH) and Lower Lows (LL)

"""

def add_market_structure(df):
    df['SwingHighPrice'] = np.where(df['SwingHigh'], df['High'], np.nan)
    df['SwingLowPrice']  = np.where(df['SwingLow'],  df['Low'],  np.nan)

    # forward-fill last swing points
    df['LastSwingHigh'] = pd.Series(df['SwingHighPrice']).ffill()
    df['LastSwingLow']  = pd.Series(df['SwingLowPrice']).ffill()

    # previous swing points (one swing back)
    prev_swing_high = df['SwingHighPrice'].dropna().shift(1)
    prev_swing_low  = df['SwingLowPrice'].dropna().shift(1)

    # map back to full index by ffill after aligning
    df['PrevSwingHigh'] = prev_swing_high.reindex(df.index).ffill()
    df['PrevSwingLow']  = prev_swing_low.reindex(df.index).ffill()

    # market structure conditions
    df['Uptrend'] = (df['LastSwingHigh'] > df['PrevSwingHigh']) & (df['LastSwingLow'] > df['PrevSwingLow'])
    df['Downtrend'] = (df['LastSwingHigh'] < df['PrevSwingHigh']) & (df['LastSwingLow'] < df['PrevSwingLow'])

    # optional single label
    df['Trend'] = np.where(df['Uptrend'], 'Uptrend',
                   np.where(df['Downtrend'], 'Downtrend', 'Range/Unclear'))
    return df


In [28]:
# Make sure ATR_14 exists (skip if you already created it)
if 'ATR_14' not in H1.columns:
    H1_tmp = H1.copy()
    prev_close = H1_tmp['Close'].shift(1)
    tr = pd.concat([
        (H1_tmp['High'] - H1_tmp['Low']),
        (H1_tmp['High'] - prev_close).abs(),
        (H1_tmp['Low'] - prev_close).abs()
    ], axis=1).max(axis=1)
    H1_tmp['ATR_14'] = tr.rolling(14).mean()
else:
    H1_tmp = H1[['Open','High','Low','Close','ATR_14']].copy()

# Apply structure pipeline to the copy
tmp = add_pivots(H1_tmp, left=3, right=3)
tmp = confirm_swings(tmp, lookahead=20, atr_col='ATR_14', atr_mult=1.0)
tmp = add_market_structure(tmp)

# Separate dataframe containing only the market-structure outputs
H1_structure = tmp[[
    'PivotHigh','PivotLow',
    'SwingHigh','SwingLow',
    'LastSwingHigh','PrevSwingHigh',
    'LastSwingLow','PrevSwingLow',
    'Uptrend','Downtrend','Trend'
]].copy()


In [29]:
# --- TESTS / VALIDATION ---

# A) Index check
assert isinstance(H1.index, pd.DatetimeIndex), "H1 index must be a DatetimeIndex to use time-based features."

# B) Column presence check
required_cols = {'Open','High','Low','Close'}
missing = required_cols - set(H1.columns)
assert not missing, f"Missing required OHLC columns: {missing}"

# C) Sanity checks on outputs
print("Rows:", len(H1_structure))
print("PivotHigh count:", int(H1_structure['PivotHigh'].sum()))
print("PivotLow count:", int(H1_structure['PivotLow'].sum()))
print("SwingHigh (confirmed) count:", int(H1_structure['SwingHigh'].sum()))
print("SwingLow (confirmed) count:", int(H1_structure['SwingLow'].sum()))

print("\nTrend distribution:")
print(H1_structure['Trend'].value_counts(dropna=False))

# D) Show a few confirmed swing rows for manual inspection
swing_rows = H1_structure.index[H1_structure['SwingHigh'] | H1_structure['SwingLow']]
print("\nSample confirmed swings (first 10):")
display(H1.loc[swing_rows[:10], ['Open','High','Low','Close']].join(H1_structure.loc[swing_rows[:10], ['SwingHigh','SwingLow','Trend']]))

# E) Quick consistency checks
# SwingHigh should be a PivotHigh; SwingLow should be a PivotLow
assert (H1_structure.loc[H1_structure['SwingHigh'], 'PivotHigh']).all(), "Found SwingHigh that is not PivotHigh."
assert (H1_structure.loc[H1_structure['SwingLow'], 'PivotLow']).all(), "Found SwingLow that is not PivotLow."

print("\n✅ Tests passed: structure features generated and basic consistency checks succeeded.")


Rows: 65156
PivotHigh count: 6351
PivotLow count: 6534
SwingHigh (confirmed) count: 6339
SwingLow (confirmed) count: 6508

Trend distribution:
Trend
Range/Unclear    28010
Uptrend          19442
Downtrend        17704
Name: count, dtype: int64

Sample confirmed swings (first 10):


,Open,High,Low,Close,SwingHigh,SwingLow,Trend
time,,,,,,,
2015-01-04 23:00:00,1188.043,1188.240,1177.321,1183.835,False,True,Range/Unclear
2015-01-05 05:00:00,1191.642,1197.704,1191.384,1196.405,True,False,Range/Unclear
2015-01-05 12:00:00,1188.946,1190.489,1186.651,1189.022,False,True,Range/Unclear
2015-01-05 20:00:00,1203.780,1207.484,1203.446,1205.780,True,False,Uptrend
2015-01-05 23:00:00,1204.183,1204.877,1200.753,1203.233,False,True,Uptrend
2015-01-06 09:00:00,1211.727,1214.254,1210.444,1213.272,True,False,Uptrend
2015-01-06 15:00:00,1209.858,1213.278,1205.295,1208.859,False,True,Uptrend
2015-01-06 16:00:00,1208.760,1223.068,1207.089,1219.623,True,False,Uptrend
2015-01-07 05:00:00,1215.103,1215.103,1211.915,1213.526,False,True,Uptrend



✅ Tests passed: structure features generated and basic consistency checks succeeded.
